# Version 6 Robustness Study

Multi-seed robustness study derived from Version 6. The notebook keeps the final participant-level PHQ-8 regression pipeline and evaluates how the baseline and Wav2Vec2 models vary across repeated training runs with different random seeds.


## Environment and Reproducibility

Imports core dependencies and fixes random seeds for reproducible experiments.


In [ ]:
# ============================================================
# ENVIRONMENT & REPRODUCIBILITY
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ---Standard library ---
import io
import time
import random
import zipfile
import warnings
from collections import Counter, defaultdict

# --- Data & numerics ---
import numpy as np
import pandas as pd

# --- PyTorch core ---------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf

# --- PyTorch Geometric ---
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, SAGEConv, global_mean_pool

# --- Transformers ---------
from transformers import Wav2Vec2Model, Wav2Vec2Processor

# --- PyTorch Lightning ---
import pytorch_lightning as pl
from pytorch_lightning import seed_everything
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import CSVLogger

# --- Data handling ---------
from torch.utils.data import Dataset, DataLoader
import requests
from bs4 import BeautifulSoup

# --- Scikit-learn ---------
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# --- Visualization ---------
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, clear_output
from tqdm import tqdm

# --- Warnings ---------------
warnings.filterwarnings("ignore", module="rich")
warnings.filterwarnings("ignore", message="In 2.9, this function")
warnings.filterwarnings("ignore", message='install "ipywidgets"')

# --- Reproducibility ------
SEED = 42
seed_everything(SEED, workers=True)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# --- Environment check ---
print("TFM environment initialized successfully.")
print(f"PyTorch     : {torch.__version__}")
print(f"Lightning   : {pl.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"VRAM used   : {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")


## Paths and Global Configuration

Defines data locations, label files, feature directories, and shared constants.


In [ ]:
# --- Global configuration ------------------------------------------------------
BASE_DIR         = "/home/arosario/depression-gnn-project/DAIC_WOZ_features"
LABELS_PATH      = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/train_split_Depression_AVEC2017.csv"
SAVE_DIR         = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/DAIC_WOZ_graphs"
WAV2VEC_SAVE_DIR = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/preprocessed_wav2vec"
GRAPHS_DIR       = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/DAIC_WOZ_graphs"
WAV2VEC_DIR      = "/home/arosario/depression-gnn-project/DAIC_WOZ_features/preprocessed_wav2vec"
CKPT_DIR         = "/home/arosario/depression-gnn-project/ckpts"
LOG_DIR          = "/home/arosario/depression-gnn-project/logs"
TMP_DIR          = "/home/arosario/depression-gnn-project/tmp"
BASE_URL         = "https://dcapswoz.ict.usc.edu/wwwdaicwoz/"


WINDOW_SIZE   = 150   # frames COVAREP por ventana de agregacin
WINDOW_FRAMES = 10     # frames CLNF por ventana del modelo
STRIDE_FRAMES = 10     # stride entre windows
BATCH_SIZE    = 32
SAMPLE_RATE   = 16000
WINDOW_MS     = 300

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = {
    0: "Sin sintomas",
    1: "Leve",
    2: "Moderado",
    3: "Moderadamente severo",
    4: "Severo"
}

# --- Create dirs ------------
for d in [SAVE_DIR, GRAPHS_DIR, WAV2VEC_DIR, CKPT_DIR, LOG_DIR, TMP_DIR]:
    os.makedirs(d, exist_ok=True)

print("Global config initialized successfully.")
print(f"Device        : {DEVICE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Window frames : {WINDOW_FRAMES}")
print(f"Stride frames : {STRIDE_FRAMES}")


## Cached CLNF and COVAREP Graph Features

This robustness notebook does not download DAIC-WOZ ZIP files or rebuild graph tensors. It reuses the cached `.pt` graph files produced for Version 6.


In [ ]:
# ============================================================
# CACHED GRAPH FEATURE VALIDATION
# ============================================================
from pathlib import Path

GRAPH_CACHE_DIR = Path(GRAPHS_DIR)
graph_cache_files = sorted(GRAPH_CACHE_DIR.glob("*_P.pt"))

if not graph_cache_files:
    raise FileNotFoundError(
        f"No cached graph tensors found in {GRAPH_CACHE_DIR}. "
        "Run the Version 6 preprocessing notebook first."
    )

sample_graphs = torch.load(graph_cache_files[0], map_location="cpu", weights_only=True)
if not sample_graphs:
    raise ValueError(f"Cached graph file is empty: {graph_cache_files[0]}")

sample_graph = sample_graphs[0]
required_graph_keys = {"x", "audio", "edge_index", "y", "phq_score"}
missing_graph_keys = required_graph_keys.difference(sample_graph.keys())
if missing_graph_keys:
    raise KeyError(f"Cached graph file {graph_cache_files[0].name} is missing keys: {sorted(missing_graph_keys)}")

print(f"[OK] Reusing cached Version 6 graph tensors from: {GRAPH_CACHE_DIR}")
print(f"[OK] Graph cache files: {len(graph_cache_files)}")
print(f"[OK] Sample participant: {graph_cache_files[0].name}")
print(f"[OK] Sample frames: {len(sample_graphs)}")
print(f"[OK] CLNF tensor shape per frame: {tuple(sample_graph['x'].shape)}")
print(f"[OK] COVAREP tensor shape per frame: {tuple(sample_graph['audio'].shape)}")


## Cached Wav2Vec2 Embeddings

This robustness notebook does not download audio files or run the Wav2Vec2 feature extractor. It reuses the cached Wav2Vec2 `.pt` embeddings produced for Version 6.


In [ ]:
# ============================================================
# CACHED WAV2VEC2 EMBEDDING VALIDATION
# ============================================================
WAV2VEC_CACHE_DIR = Path(WAV2VEC_DIR)
wav2vec_cache_files = sorted(WAV2VEC_CACHE_DIR.glob("*_P.pt"))

if not wav2vec_cache_files:
    raise FileNotFoundError(
        f"No cached Wav2Vec2 tensors found in {WAV2VEC_CACHE_DIR}. "
        "Run the Version 6 Wav2Vec2 preprocessing first."
    )

sample_wav2vec = torch.load(wav2vec_cache_files[0], map_location="cpu", weights_only=True)
if isinstance(sample_wav2vec, dict):
    sample_wav2vec = sample_wav2vec.get("embeddings", sample_wav2vec.get("embedding"))
if sample_wav2vec is None:
    raise KeyError(f"Could not find embeddings in cached Wav2Vec2 file: {wav2vec_cache_files[0].name}")

print(f"[OK] Reusing cached Version 6 Wav2Vec2 tensors from: {WAV2VEC_CACHE_DIR}")
print(f"[OK] Wav2Vec2 cache files: {len(wav2vec_cache_files)}")
print(f"[OK] Sample participant: {wav2vec_cache_files[0].name}")
print(f"[OK] Sample Wav2Vec2 tensor shape: {tuple(sample_wav2vec.shape)}")


## Cached Feature Assembly

The multimodal participant tensors are already assembled in the Version 6 cache. The following placeholder documents that no feature reconstruction is performed in this robustness notebook.


In [ ]:
# ============================================================
# FEATURE ASSEMBLY SKIPPED FOR ROBUSTNESS STUDY
# ============================================================
print("[OK] Feature assembly is skipped: using existing Version 6 .pt caches.")
print(f"[OK] Graph cache directory   : {GRAPHS_DIR}")
print(f"[OK] Wav2Vec2 cache directory: {WAV2VEC_DIR}")


## Facial Graph Topology

Builds the facial landmark graph used by the graph neural network models.


In [ ]:
# ============================================================
# SECTION 2: Facial topology (68 landmarks, bidirectional)
# ============================================================
FACIAL_EDGES = [
    # Jaw
    (0,1),(1,2),(2,3),(3,4),(4,5),(5,6),(6,7),(7,8),
    (8,9),(9,10),(10,11),(11,12),(12,13),(13,14),(14,15),(15,16),
    # Left eyebrow
    (17,18),(18,19),(19,20),(20,21),
    # Right eyebrow
    (22,23),(23,24),(24,25),(25,26),
    # Nose
    (27,28),(28,29),(29,30),
    (30,31),(31,32),(32,33),(33,34),(34,35),
    # Left eye
    (36,37),(37,38),(38,39),(39,40),(40,41),(41,36),
    # Right eye
    (42,43),(43,44),(44,45),(45,46),(46,47),(47,42),
    # Mouth outer
    (48,49),(49,50),(50,51),(51,52),(52,53),(53,54),
    (54,55),(55,56),(56,57),(57,58),(58,59),(59,48),
]

# Bidireccional (requerido por PyG)
edges_both = FACIAL_EDGES + [(b, a) for a, b in FACIAL_EDGES]
edge_index = torch.tensor(edges_both, dtype=torch.long).t().contiguous()

print(f"Edges totales (bidirectional): {edge_index.shape[1]}")

_edge_index_cache = {}

def get_expanded_edge_index(edge_index, B, W, N, device):
    """Expand a single facial graph edge index across all graphs in a mini-batch."""
    key = (B * W, N, str(device))
    if key not in _edge_index_cache:
        offsets  = torch.arange(B * W, device=device) * N
        expanded = edge_index.to(device).unsqueeze(0) + offsets.view(-1, 1, 1)
        _edge_index_cache[key] = expanded.permute(1, 0, 2).reshape(2, -1)
    return _edge_index_cache[key]

def get_batch_vec(B, W, N, device):
    """Build the PyG batch vector that assigns each facial landmark node to its graph."""
    key = (B * W, N, str(device), "bv")
    if key not in _edge_index_cache:
        _edge_index_cache[key] = torch.arange(
            B * W, device=device
        ).repeat_interleave(N)
    return _edge_index_cache[key]


## Dataset Splits, Normalization, and Augmentation

Creates participant-aware splits, normalization statistics, and data augmentation utilities.


In [ ]:
import os
import torch
import numpy as np
from collections import Counter
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedGroupKFold

# ============================================================
# Version 6: participant-level PHQ-8 regression
# CV: StratifiedGroupKFold over official train IDs
# External test: official dev IDs untouched
# ============================================================
train_ids = [
    303, 304, 305, 310, 312, 313, 315, 316, 317, 318, 319, 320, 321,
    322, 324, 325, 326, 327, 328, 330, 333, 336, 338, 339, 340, 341,
    343, 344, 345, 347, 348, 350, 351, 352, 353, 355, 356, 357, 358,
    360, 362, 363, 364, 366, 368, 369, 370, 371, 372, 374, 375, 376,
    379, 380, 383, 385, 386, 391, 392, 393, 397, 400, 401, 402, 409,
    412, 414, 415, 416, 419, 423, 425, 426, 427, 428, 429, 430, 433,
    434, 437, 441, 443, 444, 445, 446, 447, 448, 449, 454, 455, 456,
    457, 459, 463, 464, 468, 471, 473, 474, 475, 478, 479, 485, 486,
    487, 488, 491,
]

dev_ids = [
    302, 307, 331, 335, 346, 367, 377, 381, 382, 388, 389, 390, 395,
    403, 404, 406, 413, 417, 418, 420, 422, 436, 439, 440, 451, 458,
    472, 476, 477, 482, 483, 484, 489, 490, 492,
]

PHQ8_MIN = 0.0
PHQ8_MAX = 24.0
N_SPLITS = 5
ACTIVE_FOLD = 0
PARTICIPANT_WINDOWS = 32  # K windows sampled per participant per item

# V9: stronger participant-level oversampling.
# The sampler still samples participants, not windows. With augment=True,
# repeated participants produce different sampled windows across draws.
OVERSAMPLE_FACTOR = 3
SAMPLER_SEVERITY_BOOST = {
    0: 1.0,  # Sin sintomas
    1: 1.3,  # Leve
    2: 1.8,  # Moderado
    3: 3.0,  # Moderadamente severo
    4: 4.0,  # Severo
}

def phq8_to_class(score):
    """Map a PHQ-8 score to the ordinal depression severity class used in the experiments."""
    score = float(score)
    if score <= 4:    return 0
    elif score <= 9:  return 1
    elif score <= 14: return 2
    elif score <= 19: return 3
    else:             return 4

def phq8_scores_to_classes(scores):
    """Vectorize PHQ-8 severity-class conversion for a sequence of scores."""
    scores = np.asarray(scores, dtype=float)
    clipped = np.clip(scores, PHQ8_MIN, PHQ8_MAX)
    return np.array([phq8_to_class(s) for s in clipped], dtype=int)

def get_pid_score(pid, graphs_dir=GRAPHS_DIR):
    """Return the PHQ-8 score associated with a participant identifier."""
    pid_key = f"{pid}_P"
    if "df_labels" in globals() and pid_key in df_labels.index:
        return float(df_labels.loc[pid_key, "PHQ8_Score"])

    graph_path = os.path.join(graphs_dir, f"{pid_key}.pt")
    graphs = torch.load(graph_path, weights_only=True)
    g0 = graphs[0]
    if "phq_score" in g0:
        return float(torch.as_tensor(g0["phq_score"]).item())
    print(f"[WARN] {pid_key}.pt has no phq_score; falling back to y. Prefer loading df_labels before this cell.")
    return float(torch.as_tensor(g0["y"]).item())

participant_scores = {pid: get_pid_score(pid) for pid in train_ids + dev_ids}
participant_bins = {pid: phq8_to_class(score) for pid, score in participant_scores.items()}

def print_split_distribution(name, pids):
    """Print participant counts and PHQ-8 score distributions for a split."""
    counts = Counter(participant_bins[pid] for pid in pids)
    scores = np.array([participant_scores[pid] for pid in pids], dtype=float)
    print(f"{name}: n={len(pids)} score_mean={scores.mean():.2f} score_std={scores.std():.2f} range=[{scores.min():.1f}, {scores.max():.1f}]")
    for cls, cls_name in CLASS_NAMES.items():
        print(f"  {cls_name:25s}: {counts.get(cls, 0):3d} participants")

X = np.array(train_ids)
y_bins = np.array([participant_bins[pid] for pid in train_ids])
groups = np.array(train_ids)
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

CV_SPLITS = []
for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X, y_bins, groups)):
    tr = sorted(X[tr_idx].tolist())
    va = sorted(X[va_idx].tolist())
    CV_SPLITS.append({"fold": fold, "train": tr, "val": va, "test": sorted(dev_ids)})

train_final = CV_SPLITS[ACTIVE_FOLD]["train"]
val_ids = CV_SPLITS[ACTIVE_FOLD]["val"]
test_ids = sorted(dev_ids)

print("--- Version 6 split strategy: fold 0 optimisation with severity oversampling ---")
print("StratifiedGroupKFold over official train IDs. Official dev IDs are held out as external test.")
print(f"Active fold: {ACTIVE_FOLD} / {N_SPLITS - 1}")
print_split_distribution("TRAIN", train_final)
print_split_distribution("VAL", val_ids)
print_split_distribution("TEST_EXTERNAL_DEV", test_ids)

# ============================================================
# Normalisation stats, computed on fold train participants only
# ============================================================
def compute_norm_stats(pids, graphs_dir):
    """Compute feature normalization statistics from the training participants."""
    all_x, all_audio = [], []
    for pid in pids:
        path = os.path.join(graphs_dir, f"{pid}_P.pt")
        if not os.path.exists(path):
            continue
        graphs = torch.load(path, weights_only=True)
        if not graphs:
            continue
        x_t = torch.stack([g["x"] for g in graphs])
        aud_t = torch.stack([g["audio"] for g in graphs])
        x_t = torch.nan_to_num(x_t, nan=0.0, posinf=0.0, neginf=0.0)
        aud_t = torch.nan_to_num(aud_t, nan=0.0, posinf=0.0, neginf=0.0)
        all_x.append(x_t)
        all_audio.append(aud_t)

    all_x = torch.cat(all_x, dim=0)
    all_audio = torch.cat(all_audio, dim=0)
    x_mean = all_x.mean(dim=(0, 1), keepdim=True)
    x_std = all_x.std(dim=(0, 1), keepdim=True)
    x_std[x_std < 1e-5] = 1.0
    a_mean = all_audio.mean(dim=0, keepdim=True)
    a_std = all_audio.std(dim=0, keepdim=True)
    a_std[a_std < 1e-5] = 1.0
    return x_mean, x_std, a_mean, a_std


class ParticipantAugmentor:
    """Apply participant-level augmentations to facial and audio feature sequences."""
    def __init__(
        self,
        node_noise_std=0.003,
        node_drop_prob=0.01,
        edge_drop_prob=0.01,
        flip_prob=0.15,
        audio_noise_std=0.003,
        audio_feature_drop_prob=0.005,
        audio_time_mask_prob=0.03,
        audio_time_mask_max=1,
        enabled=True,
    ):
        """Initialize the object with configuration, modules, or cached experiment state."""
        self.node_noise_std = node_noise_std
        self.node_drop_prob = node_drop_prob
        self.edge_drop_prob = edge_drop_prob
        self.flip_prob = flip_prob
        self.audio_noise_std = audio_noise_std
        self.audio_feature_drop_prob = audio_feature_drop_prob
        self.audio_time_mask_prob = audio_time_mask_prob
        self.audio_time_mask_max = audio_time_mask_max
        self.enabled = enabled

    def augment_face(self, x_face, edge_index):
        """Apply stochastic noise and masking augmentation to facial landmark features."""
        if not self.enabled:
            return x_face, edge_index
        if self.node_noise_std > 0:
            x_face = x_face + torch.randn_like(x_face) * self.node_noise_std
        if self.node_drop_prob > 0:
            mask = torch.rand(x_face.shape[-2], device=x_face.device) > self.node_drop_prob
            x_face = x_face * mask.view(1, 1, -1, 1)
        if self.edge_drop_prob > 0:
            mask = torch.rand(edge_index.shape[1], device=edge_index.device) > self.edge_drop_prob
            edge_index = edge_index[:, mask]
        if torch.rand(1).item() < self.flip_prob:
            x_face = x_face.clone()
            x_face[..., 0] = -x_face[..., 0]
        return x_face, edge_index

    def augment_audio(self, x_audio):
        # x_audio: (K, W, D)
        """Apply stochastic noise and scaling augmentation to audio features."""
        if not self.enabled:
            return x_audio
        if self.audio_noise_std > 0:
            x_audio = x_audio + torch.randn_like(x_audio) * self.audio_noise_std
        if self.audio_feature_drop_prob > 0:
            feat_mask = torch.rand(x_audio.shape[-1], device=x_audio.device) > self.audio_feature_drop_prob
            x_audio = x_audio * feat_mask.view(1, 1, -1)
        if self.audio_time_mask_prob > 0 and torch.rand(1).item() < self.audio_time_mask_prob:
            mask_len = int(torch.randint(1, self.audio_time_mask_max + 1, (1,)).item())
            mask_len = min(mask_len, x_audio.shape[1])
            start = int(torch.randint(0, x_audio.shape[1] - mask_len + 1, (1,)).item())
            x_audio = x_audio.clone()
            x_audio[:, start:start + mask_len, :] = 0.0
        return x_audio


class DAICParticipantDataset(Dataset):
    """
    One item = one participant.
    Returns K temporal windows and one PHQ-8 score target.
    """
    def __init__(
        self,
        pids,
        graphs_dir,
        wav2vec_dir,
        edge_index,
        x_mean, x_std,
        a_mean, a_std,
        window_frames=10,
        stride_frames=10,
        participant_windows=32,
        augment=False,
    ):
        """Initialize the object with configuration, modules, or cached experiment state."""
        self.pids = list(pids)
        self.graphs_dir = graphs_dir
        self.wav2vec_dir = wav2vec_dir
        self.edge_index = edge_index
        self.x_mean = x_mean
        self.x_std = x_std
        self.a_mean = a_mean
        self.a_std = a_std
        self.window_frames = window_frames
        self.stride_frames = stride_frames
        self.participant_windows = participant_windows
        self.augment = augment
        self.augmentor = ParticipantAugmentor(enabled=augment)
        self.pid_cache = {}

        clnf_step_s = 1 / 30
        wav2vec_win_s = 0.3

        kept = []
        for pid in self.pids:
            graph_path = os.path.join(graphs_dir, f"{pid}_P.pt")
            wav2vec_path = os.path.join(wav2vec_dir, f"{pid}_P.pt")
            if not os.path.exists(graph_path):
                print(f"[MISSING] graph  : {pid}_P.pt")
                continue
            if not os.path.exists(wav2vec_path):
                print(f"[MISSING] wav2vec: {pid}_P.pt")
                continue

            graphs = torch.load(graph_path, weights_only=True)
            if not graphs or len(graphs) < window_frames:
                continue

            x_t = torch.stack([g["x"] for g in graphs])
            aud_t = torch.stack([g["audio"] for g in graphs])
            x_t = torch.nan_to_num(x_t, nan=0.0, posinf=0.0, neginf=0.0)
            aud_t = torch.nan_to_num(aud_t, nan=0.0, posinf=0.0, neginf=0.0)
            x_t = (x_t - x_mean) / x_std
            aud_t = (aud_t - a_mean) / a_std

            wav2vec = torch.load(wav2vec_path, weights_only=True)
            if isinstance(wav2vec, dict):
                wav2vec = wav2vec["embeddings"]
            wav2vec = torch.nan_to_num(wav2vec, nan=0.0, posinf=0.0, neginf=0.0)

            starts = list(range(0, len(graphs) - window_frames + 1, stride_frames))
            score = get_pid_score(pid)
            self.pid_cache[pid] = {
                "x": x_t,
                "audio": aud_t,
                "wav2vec": wav2vec,
                "starts": starts,
                "y": float(score),
                "y_class": int(phq8_to_class(score)),
                "n_emb": wav2vec.shape[0],
                "clnf_step": clnf_step_s,
                "wav2vec_win": wav2vec_win_s,
            }
            kept.append(pid)

        self.pids = kept
        print(f"Participants indexed: {len(self.pids)} | K windows={participant_windows} | augment={augment}")

    def __len__(self):
        """Return the number of available samples or participant groups."""
        return len(self.pids)

    def _select_starts(self, starts):
        """Select temporal window start positions for one participant sequence."""
        if self.augment:
            idx = np.random.choice(len(starts), size=self.participant_windows, replace=(len(starts) < self.participant_windows))
            return [starts[i] for i in idx]
        if len(starts) >= self.participant_windows:
            idx = np.linspace(0, len(starts) - 1, self.participant_windows).round().astype(int)
            return [starts[i] for i in idx]
        reps = int(np.ceil(self.participant_windows / len(starts)))
        return (starts * reps)[:self.participant_windows]

    def __getitem__(self, idx):
        """Return one dataset item in model-ready tensor format."""
        pid = self.pids[idx]
        cache = self.pid_cache[pid]
        selected_starts = self._select_starts(cache["starts"])

        face_windows, covarep_windows, wav2vec_windows = [], [], []
        for start in selected_starts:
            end = start + self.window_frames
            face_windows.append(cache["x"][start:end].clone())
            covarep_windows.append(cache["audio"][start:end].clone())
            indices = [
                min(int((start + j) * cache["clnf_step"] / cache["wav2vec_win"]), cache["n_emb"] - 1)
                for j in range(self.window_frames)
            ]
            wav2vec_windows.append(cache["wav2vec"][indices].clone())

        x_face = torch.stack(face_windows)       # (K, W, 68, 3)
        x_audio = torch.stack(covarep_windows)   # (K, W, 74)
        x_audio_w2v = torch.stack(wav2vec_windows)  # (K, W, 768)
        ei = self.edge_index.clone()

        x_face, ei = self.augmentor.augment_face(x_face, ei)
        x_audio = self.augmentor.augment_audio(x_audio)
        x_audio_w2v = self.augmentor.augment_audio(x_audio_w2v)

        return {
            "x_face": x_face,
            "x_audio": x_audio,
            "x_audio_w2v": x_audio_w2v,
            "edge_index": ei,
            "y": torch.tensor(cache["y"], dtype=torch.float),
            "y_class": torch.tensor(cache["y_class"], dtype=torch.long),
            "pid": pid,
        }


x_mean, x_std, a_mean, a_std = compute_norm_stats(train_final, GRAPHS_DIR)
print("Normalisation stats computed.")

train_dataset = DAICParticipantDataset(train_final, GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                                       x_mean, x_std, a_mean, a_std,
                                       window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                                       participant_windows=PARTICIPANT_WINDOWS,
                                       augment=True)
val_dataset = DAICParticipantDataset(val_ids, GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                                     x_mean, x_std, a_mean, a_std,
                                     window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                                     participant_windows=PARTICIPANT_WINDOWS,
                                     augment=False)
test_dataset = DAICParticipantDataset(test_ids, GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                                      x_mean, x_std, a_mean, a_std,
                                      window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                                      participant_windows=PARTICIPANT_WINDOWS,
                                      augment=False)


## Live Metrics Callback

Tracks training and validation metrics during PyTorch Lightning runs.


In [ ]:
class LiveMetricsLogger(Callback):
    """
    PyTorch Lightning callback that displays live

    loss, accuracy, and F1 curves for train and validation.
    Se actualiza al final de cada epoch.
    """

    def __init__(self):
        """Initialize the object with configuration, modules, or cached experiment state."""
        self.train_loss, self.val_loss = [], []
        self.train_acc,  self.val_acc  = [], []
        self.train_f1,   self.val_f1   = [], []
        self.epochs = []

    def on_train_epoch_end(self, trainer, pl_module):
        """Aggregate and log metrics at the end of a training epoch."""
        metrics = trainer.callback_metrics
        self.epochs.append(trainer.current_epoch + 1)
        self.train_loss.append(metrics.get("train_loss", torch.tensor(0)).item())
        self.train_acc.append(metrics.get("train_acc",  torch.tensor(0)).item())
        self.train_f1.append(metrics.get("train_f1",   torch.tensor(0)).item())

    def on_validation_epoch_end(self, trainer, pl_module):
        """Aggregate and log metrics at the end of a validation epoch."""
        metrics = trainer.callback_metrics

        # Ignorar sanity check (epoch 0)
        if trainer.current_epoch == 0 and trainer.global_step == 0:
            return

        self.val_loss.append(metrics.get("val_loss", torch.tensor(0)).item())
        self.val_acc.append(metrics.get("val_acc",   torch.tensor(0)).item())
        self.val_f1.append(metrics.get("val_f1",     torch.tensor(0)).item())

        self._plot()

    def _plot(self):
        """Render the currently tracked training and validation curves."""
        n = min(len(self.train_loss), len(self.val_loss))
        if n == 0:
            return

        epochs = self.epochs[:n]

        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle(f"Epoch {epochs[-1]}", fontsize=13, fontweight="bold")

        # Loss
        axes[0].plot(epochs, self.train_loss[:n], label="Train", marker="o", markersize=3)
        axes[0].plot(epochs, self.val_loss[:n],   label="Val",   marker="o", markersize=3)
        axes[0].set_title("Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Accuracy
        axes[1].plot(epochs, self.train_acc[:n], label="Train", marker="o", markersize=3)
        axes[1].plot(epochs, self.val_acc[:n],   label="Val",   marker="o", markersize=3)
        axes[1].set_title("Accuracy")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylim(0, 1)
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        # F1
        axes[2].plot(epochs, self.train_f1[:n], label="Train", marker="o", markersize=3)
        axes[2].plot(epochs, self.val_f1[:n],   label="Val",   marker="o", markersize=3)
        axes[2].set_title("F1 Score (macro)")
        axes[2].set_xlabel("Epoch")
        axes[2].set_ylim(0, 1)
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        display(fig)
        plt.close(fig)


## Participant-Grouped Sampling

Samples windows while preserving participant-level weighting.


In [ ]:
from torch.utils.data import Sampler, WeightedRandomSampler

class ParticipantGroupedSampler(Sampler):
    """
    Kept only for ablation/debugging. The main training loaders below use mixed
    batches, because grouped participant batches inflate batch-level F1 and can
    destabilise BatchNorm.
    """
    def __init__(self, dataset, shuffle=True):
        """Initialize the object with configuration, modules, or cached experiment state."""
        self.dataset = dataset
        self.shuffle = shuffle
        self.pid_to_indices = defaultdict(list)
        for idx, (pid, _) in enumerate(dataset.samples):
            self.pid_to_indices[pid].append(idx)
        self.pids = list(self.pid_to_indices.keys())

    def __iter__(self):
        """Yield sampled dataset indices for an epoch."""
        pids = self.pids.copy()
        if self.shuffle:
            random.shuffle(pids)
        for pid in pids:
            indices = self.pid_to_indices[pid].copy()
            if self.shuffle:
                random.shuffle(indices)
            yield from indices

    def __len__(self):
        """Return the number of available samples or participant groups."""
        return len(self.dataset)


def make_window_sampler_from_participant_weights(dataset, participant_weight_by_pid):
    """Create a window sampler by broadcasting participant weights to their windows."""
    sample_weights = [
        float(participant_weight_by_pid.get(pid, 1.0))
        for pid, _ in dataset.samples
    ]
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


## DataLoaders and Batch Collation

Builds batches and loaders for train, validation, and test evaluation.


In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler
from collections import Counter
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def collate_fn(batch):
    """Merge participant windows into the batch structure consumed by the models."""
    return {
        "x_face": torch.stack([b["x_face"] for b in batch]),          # (B, K, W, 68, 3)
        "x_audio": torch.stack([b["x_audio"] for b in batch]),        # (B, K, W, 74)
        "x_audio_w2v": torch.stack([b["x_audio_w2v"] for b in batch]),# (B, K, W, 768)
        "edge_index": batch[0]["edge_index"],
        "y": torch.stack([b["y"] for b in batch]).float(),
        "y_class": torch.stack([b["y_class"] for b in batch]).long(),
        "pid": [b["pid"] for b in batch],
    }


def participant_bin_counts(dataset):
    """Count participants per PHQ-8 severity bin."""
    return Counter(cache["y_class"] for cache in dataset.pid_cache.values())


def participant_score_summary(dataset):
    """Summarize participant-level PHQ-8 score statistics."""
    scores = np.array([cache["y"] for cache in dataset.pid_cache.values()], dtype=float)
    return {
        "n": len(scores),
        "mean": float(scores.mean()) if len(scores) else np.nan,
        "std": float(scores.std()) if len(scores) else np.nan,
        "min": float(scores.min()) if len(scores) else np.nan,
        "max": float(scores.max()) if len(scores) else np.nan,
    }


def make_participant_sampler(dataset, oversample_factor=OVERSAMPLE_FACTOR, severity_boost=SAMPLER_SEVERITY_BOOST):
    """
    Participant-level oversampling by PHQ-8 bin.

    V8 already used inverse-frequency participant sampling. V9 makes this more
    explicit by:
    - increasing the number of participant draws per epoch,
    - combining inverse-frequency weights with a severity boost.

    This is intentionally done at participant level. Since train_dataset has
    augment=True, repeated draws of the same participant can still select
    different K windows.
    """
    counts = participant_bin_counts(dataset)
    weights = []
    class_weight_debug = {}

    for pid in dataset.pids:
        cls = dataset.pid_cache[pid]["y_class"]
        base = 1.0 / max(counts[cls], 1)
        boost = float(severity_boost.get(cls, 1.0))
        weight = base * boost
        weights.append(weight)
        class_weight_debug[cls] = weight

    weights_t = torch.as_tensor(weights, dtype=torch.double)
    num_samples = int(len(weights) * oversample_factor)

    total_weight_by_class = {}
    for pid, weight in zip(dataset.pids, weights):
        cls = dataset.pid_cache[pid]["y_class"]
        total_weight_by_class[cls] = total_weight_by_class.get(cls, 0.0) + float(weight)
    total_weight = sum(total_weight_by_class.values())

    print("\n--- V9 participant oversampling ---")
    print(f"Oversample factor: {oversample_factor} -> train draws per epoch: {num_samples}")
    for cls, name in CLASS_NAMES.items():
        expected = num_samples * total_weight_by_class.get(cls, 0.0) / max(total_weight, 1e-12)
        print(
            f"  {name:25s}: count={counts.get(cls, 0):3d} "
            f"boost={severity_boost.get(cls, 1.0):.2f} "
            f"expected_draws/epoch={expected:6.1f}"
        )

    return WeightedRandomSampler(weights_t, num_samples=num_samples, replacement=True)


def make_loaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE, use_balanced_sampler=True):
    """Create train, validation, and test DataLoaders with the configured samplers."""
    print("\n--- Participant PHQ-8 score/bin distribution ---")
    for split_name, ds in [("TRAIN", train_ds), ("VAL", val_ds), ("TEST", test_ds)]:
        counts = participant_bin_counts(ds)
        summary = participant_score_summary(ds)
        print(f"{split_name}: n={summary['n']} score_mean={summary['mean']:.2f} score_std={summary['std']:.2f} range=[{summary['min']:.1f}, {summary['max']:.1f}]")
        for cls, name in CLASS_NAMES.items():
            print(f"  {name:25s}: {counts.get(cls, 0):3d} participants")

    sampler = make_participant_sampler(train_ds) if use_balanced_sampler else None
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False if sampler is not None else True,
        collate_fn=collate_fn,
        num_workers=4,
        pin_memory=True,
    )
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=4, pin_memory=True)

    print(f"\n[OK] Train : {len(train_ds)} participants (augment=True)")
    print(f"     Val   : {len(val_ds)} participants (augment=False)")
    print(f"     Test  : {len(test_ds)} participants (augment=False)")
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(train_dataset, val_dataset, test_dataset, batch_size=BATCH_SIZE)


## Regression Model Definitions

Defines PHQ-8 regression models that combine facial graph, acoustic, and metadata features.


In [ ]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch_geometric.nn import GCNConv, SAGEConv, global_mean_pool
from sklearn.metrics import mean_absolute_error, f1_score, root_mean_squared_error


def get_expanded_edge_index(edge_index, n_graphs, n_edges_per_graph, n_nodes, device):
    """Expand a single facial graph edge index across all graphs in a mini-batch."""
    offsets = torch.arange(n_graphs, device=device) * n_nodes
    ei = edge_index.to(device).unsqueeze(0).expand(n_graphs, -1, -1)
    ei = ei + offsets.view(n_graphs, 1, 1)
    return ei.reshape(2, -1)


def clip_phq8(preds):
    """Clamp predicted PHQ-8 values to the valid questionnaire score range."""
    return np.clip(np.asarray(preds, dtype=float), PHQ8_MIN, PHQ8_MAX)


def regression_metrics(labels, preds):
    """Compute regression and discretized classification metrics for PHQ-8 predictions."""
    labels = np.asarray(labels, dtype=float)
    preds = clip_phq8(preds)
    mae = mean_absolute_error(labels, preds)
    rmse = root_mean_squared_error(labels, preds)
    pseudo_f1 = f1_score(
        phq8_scores_to_classes(labels),
        phq8_scores_to_classes(preds),
        average="macro",
        zero_division=0,
    )
    return float(mae), float(rmse), float(pseudo_f1)


def phq8_bin_loss_weights(y_class, device):
    """
    Soft severity weighting for PHQ-8 regression.
    This avoids extreme inverse-frequency weights while still making errors
    on moderate/severe participants count more than errors on majority bins.
    """
    weights = torch.tensor([1.0, 1.2, 1.5, 2.0, 2.5], dtype=torch.float, device=device)
    return weights[y_class]


class AttentionPooling(nn.Module):
    """
    Learns a participant embedding from K window embeddings.
    The model still predicts once per participant, but can give more weight
    to the windows that are most informative for PHQ-8 severity.
    """
    def __init__(self, input_dim, hidden_dim=None, dropout=0.0):
        """Initialize the object with configuration, modules, or cached experiment state."""
        super().__init__()
        hidden_dim = hidden_dim or max(8, input_dim // 2)
        self.scorer = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, window_emb):
        """Compute PHQ-8 predictions from a batch of multimodal features."""
        scores = self.scorer(window_emb).squeeze(-1)  # (B, K)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (window_emb * weights).sum(dim=1)


class ParticipantRegressionMixin:
    """Shared participant-level PHQ-8 regression helpers for Lightning modules."""
    def _start_epoch_buffers(self, stage):
        """Reset temporary prediction buffers at the beginning of an epoch."""
        setattr(self, f"{stage}_preds_epoch", [])
        setattr(self, f"{stage}_labels_epoch", [])

    def on_train_epoch_start(self):
        """Reset train buffers before a new training epoch starts."""
        self._start_epoch_buffers("train")

    def on_validation_epoch_start(self):
        """Reset validation buffers before a new validation epoch starts."""
        self._start_epoch_buffers("val")

    def _log_epoch_metrics(self, stage):
        """Compute and log buffered epoch-level regression metrics."""
        labels = np.array(getattr(self, f"{stage}_labels_epoch"), dtype=float)
        preds = np.array(getattr(self, f"{stage}_preds_epoch"), dtype=float)
        if len(labels) == 0:
            return
        mae, rmse, pseudo_f1 = regression_metrics(labels, preds)
        self.log(f"{stage}_participant_mae", mae, prog_bar=(stage == "val"))
        self.log(f"{stage}_participant_rmse", rmse, prog_bar=(stage == "val"))
        self.log(f"{stage}_participant_pseudo_macro_f1", pseudo_f1, prog_bar=(stage == "val"))

    def on_train_epoch_end(self):
        """Aggregate and log metrics at the end of a training epoch."""
        self._log_epoch_metrics("train")

    def on_validation_epoch_end(self):
        """Aggregate and log metrics at the end of a validation epoch."""
        self._log_epoch_metrics("val")

    def _shared_step(self, batch, stage, audio_key):
        """Run the common model step used by training and validation."""
        x_face = batch["x_face"].to(self.device)
        x_audio = batch[audio_key].to(self.device)
        edge_index = batch["edge_index"].to(self.device)
        y = batch["y"].to(self.device).float()
        y_class = batch["y_class"].to(self.device).long()

        pred = self(x_face, x_audio, edge_index).squeeze(-1)
        per_sample_loss = F.smooth_l1_loss(pred, y, reduction="none")
        weights = phq8_bin_loss_weights(y_class, self.device)
        weights = weights / weights.mean().clamp_min(1e-6)
        loss = (per_sample_loss * weights).mean()

        bs = y.shape[0]
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=bs)
        self.log(f"{stage}_weighted_loss", loss, prog_bar=False, on_step=False, on_epoch=True, batch_size=bs)
        getattr(self, f"{stage}_preds_epoch").extend(pred.detach().cpu().numpy().tolist())
        getattr(self, f"{stage}_labels_epoch").extend(y.detach().cpu().numpy().tolist())
        return loss


class BaselineParticipantPHQ8Regressor(ParticipantRegressionMixin, pl.LightningModule):
    """Participant-level baseline graph regressor for PHQ-8 prediction."""
    def __init__(self, node_in=3, audio_in=74, gcn_hidden=16,
                 mlp_hidden=16, gru_hidden=32, dropout=0.20,
                 lr=3e-4, weight_decay=0.001):
        """Initialize the object with configuration, modules, or cached experiment state."""
        super().__init__()
        self.save_hyperparameters()
        self.gcn1 = GCNConv(node_in, gcn_hidden)
        self.bn_g1 = nn.BatchNorm1d(gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)
        self.bn_face = nn.BatchNorm1d(gcn_hidden)
        self.audio_mlp = nn.Sequential(
            nn.Linear(audio_in, mlp_hidden),
            nn.BatchNorm1d(mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(gcn_hidden + mlp_hidden, gru_hidden, num_layers=1, batch_first=True)
        self.participant_pool = AttentionPooling(gru_hidden, hidden_dim=max(8, gru_hidden // 2), dropout=dropout)
        self.regressor = nn.Sequential(nn.Dropout(dropout), nn.Linear(gru_hidden, 1))

    def _encode_faces_gcn(self, x_face, edge_index):
        """Encode facial landmark windows with the GCN branch."""
        BKW, W, N, C = x_face.shape
        x_flat = x_face.reshape(BKW * W * N, C)
        batch_vec = torch.arange(BKW * W, device=x_face.device).repeat_interleave(N)
        ei_flat = get_expanded_edge_index(edge_index, BKW * W, 1, N, x_face.device)
        h = F.relu(self.bn_g1(self.gcn1(x_flat, ei_flat)))
        h = F.relu(self.bn_face(self.gcn2(h, ei_flat)))
        return global_mean_pool(h, batch_vec).view(BKW, W, -1)

    def forward(self, x_face, x_audio, edge_index):
        """Compute PHQ-8 predictions from a batch of multimodal features."""
        B, K, W, N, C = x_face.shape
        x_face_f = x_face.reshape(B * K, W, N, C)
        x_audio_f = x_audio.reshape(B * K, W, -1)
        h_face = self._encode_faces_gcn(x_face_f, edge_index)
        h_audio = self.audio_mlp(x_audio_f.reshape(B * K * W, -1)).view(B * K, W, -1)
        h = torch.cat([h_face, h_audio], dim=-1)
        self.gru.flatten_parameters()
        _, h_n = self.gru(h)
        window_emb = h_n[-1].view(B, K, -1)
        participant_emb = self.participant_pool(window_emb)
        return self.regressor(participant_emb)

    def training_step(self, batch, batch_idx):
        """Run one optimization step and log training loss."""
        return self._shared_step(batch, "train", "x_audio")

    def validation_step(self, batch, batch_idx):
        """Run one validation step and log validation loss."""
        return self._shared_step(batch, "val", "x_audio")

    def configure_optimizers(self):
        """Create the optimizer and optional learning-rate scheduler."""
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "monitor": "val_participant_mae", "interval": "epoch"}}


class Wav2VecParticipantPHQ8Regressor(ParticipantRegressionMixin, pl.LightningModule):
    """Participant-level regressor that fuses graph features with Wav2Vec embeddings."""
    def __init__(self, node_in=3, wav2vec_dim=768, audio_proj=32,
                 sage_hidden=16, gru_hidden=64, dropout=0.20,
                 lr=3e-4, weight_decay=0.001):
        """Initialize the object with configuration, modules, or cached experiment state."""
        super().__init__()
        self.save_hyperparameters()
        self.sage1 = SAGEConv(node_in, sage_hidden)
        self.bn_s1 = nn.BatchNorm1d(sage_hidden)
        self.sage2 = SAGEConv(sage_hidden, sage_hidden)
        self.bn_face = nn.BatchNorm1d(sage_hidden)
        self.audio_proj = nn.Sequential(
            nn.Linear(wav2vec_dim, audio_proj),
            nn.BatchNorm1d(audio_proj),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.gru = nn.GRU(sage_hidden + audio_proj, gru_hidden, num_layers=1, batch_first=True)
        self.participant_pool = AttentionPooling(gru_hidden, hidden_dim=max(8, gru_hidden // 2), dropout=dropout)
        self.regressor = nn.Sequential(nn.Dropout(dropout), nn.Linear(gru_hidden, 1))

    def _encode_faces_sage(self, x_face, edge_index):
        """Encode facial landmark windows with the GraphSAGE branch."""
        BKW, W, N, C = x_face.shape
        x_flat = x_face.reshape(BKW * W * N, C)
        batch_vec = torch.arange(BKW * W, device=x_face.device).repeat_interleave(N)
        ei_flat = get_expanded_edge_index(edge_index, BKW * W, 1, N, x_face.device)
        h = F.relu(self.bn_s1(self.sage1(x_flat, ei_flat)))
        h = F.relu(self.sage2(h, ei_flat))
        h_bn = self.bn_face(global_mean_pool(h, batch_vec))
        return h_bn.view(BKW, W, -1)

    def forward(self, x_face, x_audio, edge_index):
        """Compute PHQ-8 predictions from a batch of multimodal features."""
        B, K, W, N, C = x_face.shape
        x_face_f = x_face.reshape(B * K, W, N, C)
        x_audio_f = x_audio.reshape(B * K, W, -1)
        h_face = self._encode_faces_sage(x_face_f, edge_index)
        h_audio = self.audio_proj(x_audio_f.reshape(B * K * W, -1)).view(B * K, W, -1)
        h = torch.cat([h_face, h_audio], dim=-1)
        self.gru.flatten_parameters()
        _, h_n = self.gru(h)
        window_emb = h_n[-1].view(B, K, -1)
        participant_emb = self.participant_pool(window_emb)
        return self.regressor(participant_emb)

    def training_step(self, batch, batch_idx):
        """Run one optimization step and log training loss."""
        return self._shared_step(batch, "train", "x_audio_w2v")

    def validation_step(self, batch, batch_idx):
        """Run one validation step and log validation loss."""
        return self._shared_step(batch, "val", "x_audio_w2v")

    def configure_optimizers(self):
        """Create the optimizer and optional learning-rate scheduler."""
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "monitor": "val_participant_mae", "interval": "epoch"}}


BaselinePHQ8Regressor = BaselineParticipantPHQ8Regressor
Wav2VecPHQ8Regressor = Wav2VecParticipantPHQ8Regressor
BaselineGRUModel = BaselineParticipantPHQ8Regressor
Wav2VecGRUModel = Wav2VecParticipantPHQ8Regressor


## Participant Weighting

Computes class/bin weights to reduce imbalance across PHQ-8 score ranges.


In [ ]:
from collections import Counter

def compute_participant_bin_weights(dataset):
    """Compute participant weights from PHQ-8 bin frequencies."""
    counts = Counter(cache["y_class"] for cache in dataset.pid_cache.values())
    weights = {pid: 1.0 / max(counts[dataset.pid_cache[pid]["y_class"]], 1) for pid in dataset.pids}
    print("Participant bin counts in train:", dict(counts))
    print("These weights are used by the participant sampler, not by the regression loss.")
    return weights

participant_weight_by_pid = compute_participant_bin_weights(train_dataset)
class_weights_tensor = participant_weight_by_pid


## Training Orchestration

Constructs models and launches the experiment training loop.


In [ ]:
# ============================================================
# Version 6 TRAINING: participant-level reduced PHQ-8 regression
# Models: Baseline + Wav2Vec
# ============================================================
import os
import wandb
from pytorch_lightning.loggers import WandbLogger, CSVLogger


def build_model(model_name):
    """Instantiate the requested PHQ-8 regression model variant."""
    if model_name == "baseline":
        return BaselineParticipantPHQ8Regressor(
            node_in=3, audio_in=74, gcn_hidden=16,
            mlp_hidden=16, gru_hidden=32,
            dropout=0.20, lr=3e-4, weight_decay=0.001,
        )
    if model_name == "wav2vec":
        return Wav2VecParticipantPHQ8Regressor(
            node_in=3, wav2vec_dim=768, audio_proj=32,
            sage_hidden=16, gru_hidden=64,
            dropout=0.20, lr=3e-4, weight_decay=0.001,
        )
    raise ValueError(f"Unknown model: {model_name}")


def train_all_models(train_loader, val_loader, fold=ACTIVE_FOLD):
    """Train every configured model variant and collect the resulting artifacts."""
    training_group = f"tfmv9_participant_phq8reg_fold{fold}_{wandb.util.generate_id()}"
    results = {}

    for model_name in ["baseline", "wav2vec"]:
        run = wandb.init(
            project="depression-gnn",
            group=training_group,
            name=f"tfmv9_{model_name}_participant_fold{fold}",
            config={
                "version": "Version 6",
                "model": model_name,
                "objective": "PHQ8_score_regression",
                "training_unit": "participant",
                "aggregation": "attention_pool_K_windows",
                "participant_windows": PARTICIPANT_WINDOWS,
                "loss": "weighted_smooth_l1_by_phq8_bin",
                "monitor": "val_participant_mae",
                "split_strategy": "StratifiedGroupKFold_train__external_dev_test",
                "fold": fold,
                "n_splits": N_SPLITS,
                "architecture_version": "v9_v8_plus_severity_participant_oversampling",
                "audio_augmentation": "reduced_noise_feature_dropout_temporal_mask",
                "loss_bin_weights": [1.0, 1.2, 1.5, 2.0, 2.5],
                "optimisation_scope": "single_fold0_test_participant_oversampling",
                "participant_oversampling": True,
                "oversample_factor": OVERSAMPLE_FACTOR,
                "sampler_severity_boost": SAMPLER_SEVERITY_BOOST,
                "window_frames": WINDOW_FRAMES,
                "stride_frames": STRIDE_FRAMES,
                "batch_size": BATCH_SIZE,
                "seed": SEED,
                "target": "PHQ8_Score",
            },
        )

        model = build_model(model_name)
        run.config.update({
            f"{model_name}_hparams": dict(model.hparams),
            f"{model_name}_n_params": sum(p.numel() for p in model.parameters()),
        })

        print(f"\n[INFO] Training Version 6 participant-level regressor: {model_name} | fold={fold}")
        checkpoint_callback = pl.callbacks.ModelCheckpoint(
            dirpath=CKPT_DIR,
            monitor="val_participant_mae",
            mode="min",
            save_top_k=1,
            filename=f"tfmv9-{model_name}-fold{fold}-{{epoch:02d}}-mae{{val_participant_mae:.3f}}",
            auto_insert_metric_name=False,
        )
        early_stopping = pl.callbacks.EarlyStopping(
            monitor="val_participant_mae",
            patience=8,
            mode="min",
            min_delta=0.01,
            verbose=True,
        )

        wandb_logger = WandbLogger(experiment=run, prefix=model_name)
        csv_logger = CSVLogger(save_dir=LOG_DIR, name=f"tfmv9_{model_name}_fold{fold}")
        precision = "16-mixed" if torch.cuda.is_available() else "32"
        trainer = pl.Trainer(
            max_epochs=60,
            accelerator="auto",
            devices=1,
            log_every_n_steps=5,
            logger=[wandb_logger, csv_logger],
            callbacks=[checkpoint_callback, early_stopping],
            gradient_clip_val=1.0,
            precision=precision,
        )
        trainer.fit(model, train_loader, val_loader)

        best_ckpt = checkpoint_callback.best_model_path
        best_mae = checkpoint_callback.best_model_score
        if best_ckpt and os.path.exists(best_ckpt):
            artifact = wandb.Artifact(
                name=f"tfmv9_{model_name}_fold{fold}_ckpt",
                type="model",
                metadata={"model": model_name, "fold": fold, "val_participant_mae": float(best_mae)},
            )
            artifact.add_file(best_ckpt)
            run.log_artifact(artifact)
            print(f"[WANDB] Artifact logged: {best_ckpt}")

        print(f"\n-- Version 6 {model_name} fold {fold} --")
        print(f"Best val participant MAE : {float(best_mae):.4f}")
        print(f"Checkpoint               : {best_ckpt}")

        results[model_name] = {
            "model": model,
            "trainer": trainer,
            "ckpt": checkpoint_callback,
            "best_mae": float(best_mae),
            "best_ckpt": best_ckpt,
        }
        wandb.finish()

    return results, training_group


# Single-run training is intentionally not launched in this robustness notebook.
# Use run_seed_robustness_study(...) below to train repeated runs across seeds.


## Evaluation Utilities

Collects predictions, computes metrics, and writes evaluation artifacts.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, root_mean_squared_error
from collections import Counter
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns


def bootstrap_regression_ci(labels, preds, metric_fn, n_boot=2000, ci=95, seed=SEED):
    """Estimate confidence intervals for regression metrics with bootstrap resampling."""
    labels = np.asarray(labels, dtype=float)
    preds = np.asarray(preds, dtype=float)
    if len(labels) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(labels), len(labels))
        scores.append(metric_fn(labels[idx], preds[idx]))
    alpha = (100 - ci) / 2
    return float(metric_fn(labels, preds)), float(np.percentile(scores, alpha)), float(np.percentile(scores, 100 - alpha))


def collect_scores(model, loader, device):
    """Run inference and collect predictions, targets, and participant identifiers."""
    model.eval()
    all_preds, all_labels, all_pids = [], [], []
    with torch.no_grad():
        for batch in loader:
            x_face = batch["x_face"].to(device)
            edge_index = batch["edge_index"].to(device)
            y = batch["y"].to(device).float()
            if isinstance(model, Wav2VecParticipantPHQ8Regressor):
                x_audio = batch["x_audio_w2v"].to(device)
            else:
                x_audio = batch["x_audio"].to(device)
            preds = model(x_face, x_audio, edge_index).squeeze(-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_pids.extend(batch["pid"])
    return np.array(all_preds, dtype=float), np.array(all_labels, dtype=float), all_pids


def mae_metric(y, p):
    """Compute mean absolute error for PHQ-8 predictions."""
    return mean_absolute_error(y, clip_phq8(p))


def rmse_metric(y, p):
    """Compute root mean squared error for PHQ-8 predictions."""
    return root_mean_squared_error(y, clip_phq8(p))


def pseudo_f1_metric(y, p):
    """Compute F1 after discretizing PHQ-8 regression outputs into severity classes."""
    return f1_score(phq8_scores_to_classes(y), phq8_scores_to_classes(clip_phq8(p)), average="macro", zero_division=0)


def regression_report(labels, preds):
    """Build a compact regression report from predictions and targets."""
    preds_clip = clip_phq8(preds)
    return {
        "mae": float(mean_absolute_error(labels, preds_clip)),
        "rmse": float(root_mean_squared_error(labels, preds_clip)),
        "pseudo_macro_f1": float(pseudo_f1_metric(labels, preds_clip)),
    }


def report_to_wandb_table(report):
    """Convert an evaluation report into a W&B table."""
    rows = []
    for cls_id, cls_name in CLASS_NAMES.items():
        metrics = report.get(cls_name, {})
        rows.append([cls_id, cls_name, metrics.get("precision", 0.0), metrics.get("recall", 0.0), metrics.get("f1-score", 0.0), metrics.get("support", 0)])
    return wandb.Table(columns=["class_id", "class", "precision", "recall", "f1", "support"], data=rows)


def save_confusion_matrix(labels, preds, title, path):
    """Save a confusion matrix for discretized PHQ-8 severity predictions."""
    true_cls = phq8_scores_to_classes(labels)
    pred_cls = phq8_scores_to_classes(clip_phq8(preds))
    cm = confusion_matrix(true_cls, pred_cls, labels=list(CLASS_NAMES.keys()))
    cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=list(CLASS_NAMES.values()), yticklabels=list(CLASS_NAMES.values()), ax=axes[0])
    axes[0].set_title("Counts")
    axes[0].set_xlabel("Predicted PHQ bin")
    axes[0].set_ylabel("True PHQ bin")
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, xticklabels=list(CLASS_NAMES.values()), yticklabels=list(CLASS_NAMES.values()), ax=axes[1])
    axes[1].set_title("Normalised recall")
    axes[1].set_xlabel("Predicted PHQ bin")
    axes[1].set_ylabel("True PHQ bin")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def log_regression_eval_to_wandb(run, model_name, loader_name, labels, preds, report, class_report, cm_path, ci=None):
    """Log regression metrics, intervals, and plots to Weights & Biases."""
    prefix = f"{model_name}/{loader_name}/participant"
    payload = {
        f"{prefix}/mae": report["mae"],
        f"{prefix}/rmse": report["rmse"],
        f"{prefix}/pseudo_macro_f1": report["pseudo_macro_f1"],
        f"{prefix}/pred_min": float(np.min(preds)),
        f"{prefix}/pred_mean": float(np.mean(preds)),
        f"{prefix}/pred_max": float(np.max(preds)),
        f"{prefix}/pred_std": float(np.std(preds)),
        f"{prefix}/label_min": float(np.min(labels)),
        f"{prefix}/label_mean": float(np.mean(labels)),
        f"{prefix}/label_max": float(np.max(labels)),
        f"{prefix}/label_std": float(np.std(labels)),
        f"{prefix}/per_class_pseudo": report_to_wandb_table(class_report),
        f"{prefix}/pseudo_confusion_matrix": wandb.Image(cm_path),
    }
    if ci is not None:
        payload[f"{prefix}/mae_ci_low"] = ci["mae"][1]
        payload[f"{prefix}/mae_ci_high"] = ci["mae"][2]
        payload[f"{prefix}/rmse_ci_low"] = ci["rmse"][1]
        payload[f"{prefix}/rmse_ci_high"] = ci["rmse"][2]
        payload[f"{prefix}/pseudo_macro_f1_ci_low"] = ci["pseudo_f1"][1]
        payload[f"{prefix}/pseudo_macro_f1_ci_high"] = ci["pseudo_f1"][2]
    run.log(payload)


def evaluate_model(model, loader, loader_name, model_name, device, plots_dir=None, run=None, training_group=None, n_boot=2000):
    """Evaluate one trained model on a DataLoader and return metrics/artifacts."""
    preds, labels, pids = collect_scores(model, loader, device)
    report = regression_report(labels, preds)
    print(f"\n{model_name} - {loader_name} - PARTICIPANT LEVEL ({len(labels)} participants)")
    print(f"MAE={report['mae']:.4f} | RMSE={report['rmse']:.4f} | pseudo macro F1={report['pseudo_macro_f1']:.4f}")
    print(
        f"Pred PHQ8 stats: min={preds.min():.2f}, mean={preds.mean():.2f}, "
        f"max={preds.max():.2f}, std={preds.std():.2f}"
    )
    print(
        f"True PHQ8 stats: min={labels.min():.2f}, mean={labels.mean():.2f}, "
        f"max={labels.max():.2f}, std={labels.std():.2f}"
    )
    ci = {
        "mae": bootstrap_regression_ci(labels, preds, mae_metric, n_boot=n_boot),
        "rmse": bootstrap_regression_ci(labels, preds, rmse_metric, n_boot=n_boot),
        "pseudo_f1": bootstrap_regression_ci(labels, preds, pseudo_f1_metric, n_boot=n_boot),
    }
    print(f"Participant MAE bootstrap 95% CI: {ci['mae'][0]:.4f} [{ci['mae'][1]:.4f}, {ci['mae'][2]:.4f}]")
    print(f"Participant RMSE bootstrap 95% CI: {ci['rmse'][0]:.4f} [{ci['rmse'][1]:.4f}, {ci['rmse'][2]:.4f}]")
    print(f"Participant pseudo macro F1 bootstrap 95% CI: {ci['pseudo_f1'][0]:.4f} [{ci['pseudo_f1'][1]:.4f}, {ci['pseudo_f1'][2]:.4f}]")

    true_cls = phq8_scores_to_classes(labels)
    pred_cls = phq8_scores_to_classes(clip_phq8(preds))
    class_report = classification_report(true_cls, pred_cls, labels=list(CLASS_NAMES.keys()), target_names=list(CLASS_NAMES.values()), zero_division=0, digits=4, output_dict=True)
    print("\nPseudo-classification report from binned participant PHQ-8 predictions:")
    print(classification_report(true_cls, pred_cls, labels=list(CLASS_NAMES.keys()), target_names=list(CLASS_NAMES.values()), zero_division=0, digits=4))

    if run is not None:
        if plots_dir is None:
            plots_dir = os.path.join("/home/arosario/depression-gnn-project/plots", "wandb_eval")
        os.makedirs(plots_dir, exist_ok=True)
        safe_model = str(model_name).replace(" ", "_")
        safe_loader = str(loader_name).replace(" ", "_")
        cm_path = os.path.join(plots_dir, f"{safe_model}_{safe_loader}_participant_pseudo_cm.png")
        save_confusion_matrix(labels, preds, f"{model_name} - {loader_name} - participant pseudo bins", cm_path)
        log_regression_eval_to_wandb(run, model_name, loader_name, labels, preds, report, class_report, cm_path, ci=ci)

    return {
        "participant_preds": preds,
        "participant_labels": labels,
        "participant_pids": pids,
        "participant_mae": report["mae"],
        "participant_rmse": report["rmse"],
        "participant_pseudo_macro_f1": report["pseudo_macro_f1"],
        "participant_ci": ci,
    }


## Experiment Evaluation and Cross-Validation

Runs held-out evaluation and grouped cross-validation workflows.


In [ ]:
import datetime
import os
from pytorch_lightning.loggers import WandbLogger, CSVLogger


def safe_wandb_init(**kwargs):
    """Initialize a W&B run while tolerating disabled or unavailable tracking."""
    try:
        if wandb.run is not None:
            wandb.finish()
    except Exception as exc:
        print(f"[WARN] wandb.finish() before init failed: {exc}")
    try:
        wandb.teardown()
    except Exception:
        pass
    kwargs.setdefault("reinit", True)
    try:
        kwargs.setdefault("settings", wandb.Settings(start_method="thread"))
    except Exception:
        pass
    return wandb.init(**kwargs)


def evaluate_all_models(ckpt_paths, val_loader, test_loader, device, plots_dir, training_group):
    """Evaluate all trained model variants and summarize their performance."""
    models_dict = {
        "Baseline": BaselineParticipantPHQ8Regressor.load_from_checkpoint(ckpt_paths["baseline"], strict=False).to(device),
        "Wav2Vec": Wav2VecParticipantPHQ8Regressor.load_from_checkpoint(ckpt_paths["wav2vec"], strict=False).to(device),
    }
    run = safe_wandb_init(
        project="depression-gnn",
        group=training_group,
        name="tfmv9_participant_evaluation",
        config={
            "version": "Version 6",
            "objective": "PHQ8_score_regression",
            "training_unit": "participant",
            "model_set": "baseline_wav2vec_reduced",
            "split_strategy": "StratifiedGroupKFold_train__external_dev_test",
            "metrics": "MAE_RMSE_pseudo_macro_f1",
            "participant_windows": PARTICIPANT_WINDOWS,
            "bootstrap_repeats": 2000,
        },
    )
    all_results = {}
    for model_name, model in models_dict.items():
        all_results[(model_name, "val")] = evaluate_model(model, val_loader, "val", model_name, device, plots_dir=plots_dir, run=run, training_group=training_group)
        all_results[(model_name, "test")] = evaluate_model(model, test_loader, "test_external_dev", model_name, device, plots_dir=plots_dir, run=run, training_group=training_group)
    wandb.finish()
    return all_results


def build_fold_datasets(fold):
    """Create train and validation datasets for one grouped cross-validation fold."""
    split = CV_SPLITS[fold]
    fold_x_mean, fold_x_std, fold_a_mean, fold_a_std = compute_norm_stats(split["train"], GRAPHS_DIR)
    fold_train_ds = DAICParticipantDataset(split["train"], GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                                           fold_x_mean, fold_x_std, fold_a_mean, fold_a_std,
                                           window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                                           participant_windows=PARTICIPANT_WINDOWS, augment=True)
    fold_val_ds = DAICParticipantDataset(split["val"], GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                                         fold_x_mean, fold_x_std, fold_a_mean, fold_a_std,
                                         window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                                         participant_windows=PARTICIPANT_WINDOWS, augment=False)
    fold_test_ds = DAICParticipantDataset(test_ids, GRAPHS_DIR, WAV2VEC_DIR, edge_index,
                                          fold_x_mean, fold_x_std, fold_a_mean, fold_a_std,
                                          window_frames=WINDOW_FRAMES, stride_frames=STRIDE_FRAMES,
                                          participant_windows=PARTICIPANT_WINDOWS, augment=False)
    return fold_train_ds, fold_val_ds, fold_test_ds


def run_stratified_group_kfold(model_names=("baseline", "wav2vec"), n_splits=N_SPLITS):
    """Run stratified grouped K-fold validation by participant."""
    cv_rows = []
    for fold in range(n_splits):
        print(f"\n========== Version 6 fold {fold}/{n_splits - 1} ==========")
        fold_train_ds, fold_val_ds, fold_test_ds = build_fold_datasets(fold)
        fold_train_loader, fold_val_loader, fold_test_loader = make_loaders(fold_train_ds, fold_val_ds, fold_test_ds, batch_size=BATCH_SIZE)
        fold_results, fold_group = train_all_models(fold_train_loader, fold_val_loader, fold=fold)
        for model_name in model_names:
            model = fold_results[model_name]["model"].to(DEVICE)
            eval_res = evaluate_model(model, fold_val_loader, f"fold{fold}_val", model_name, DEVICE)
            cv_rows.append({
                "fold": fold,
                "model": model_name,
                "val_participant_mae": eval_res["participant_mae"],
                "val_participant_rmse": eval_res["participant_rmse"],
                "val_participant_pseudo_macro_f1": eval_res["participant_pseudo_macro_f1"],
                "training_group": fold_group,
                "checkpoint": fold_results[model_name]["best_ckpt"],
            })
    cv_df = pd.DataFrame(cv_rows)
    display(cv_df)
    print("\nVersion 6 StratifiedGroupKFold summary:")
    summary = cv_df.groupby("model")[["val_participant_mae", "val_participant_rmse", "val_participant_pseudo_macro_f1"]].agg(["mean", "std", "count"])
    print(summary)
    return cv_df


## Robustness Study Design

This section repeats the final Version 6 training protocol across multiple random seeds. By default, the participant split is kept fixed to the active fold from Version 6, so the reported variability mainly reflects stochasticity from model initialization, sampling, augmentation, and optimization rather than changes in the train/validation partition.


In [ ]:
# ============================================================
# MULTI-SEED ROBUSTNESS STUDY
# ============================================================
import random
import datetime
from pathlib import Path

ROBUSTNESS_SEEDS = [7, 42, 123, 2024, 3407]
ROBUSTNESS_MODEL_NAMES = ["baseline", "wav2vec"]
ROBUSTNESS_MAX_EPOCHS = 60
ROBUSTNESS_PATIENCE = 8
ROBUSTNESS_SPLIT_MODE = "fixed_fold0"  # Options: "fixed_fold0" or "seeded_fold0"
ROBUSTNESS_USE_WANDB = True
ROBUSTNESS_WANDB_PROJECT = "depression-gnn"
ROBUSTNESS_WANDB_GROUP = "version6_multiseed_robustness"
ROBUSTNESS_OUTPUT_DIR = Path("/home/arosario/depression-gnn-project/robustness")
ROBUSTNESS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def set_experiment_seed(seed):
    """Set all relevant random seeds before building loaders and models."""
    seed = int(seed)
    seed_everything(seed, workers=True)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def make_cv_splits_for_seed(split_seed):
    """Create StratifiedGroupKFold splits for a requested split seed."""
    X = np.array(train_ids)
    y_bins = np.array([participant_bins[pid] for pid in train_ids])
    groups = np.array(train_ids)
    sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=int(split_seed))
    splits = []
    for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X, y_bins, groups)):
        splits.append({
            "fold": fold,
            "train": sorted(X[tr_idx].tolist()),
            "val": sorted(X[va_idx].tolist()),
            "test": sorted(dev_ids),
        })
    return splits


def get_robustness_split(seed, split_mode=ROBUSTNESS_SPLIT_MODE, fold=ACTIVE_FOLD):
    """Return the participant split used for one robustness run."""
    if split_mode == "fixed_fold0":
        split = CV_SPLITS[fold]
        return {
            "fold": fold,
            "train": list(split["train"]),
            "val": list(split["val"]),
            "test": list(split["test"]),
            "split_seed": SEED,
            "split_mode": split_mode,
        }
    if split_mode == "seeded_fold0":
        split = make_cv_splits_for_seed(seed)[fold]
        return {
            "fold": fold,
            "train": list(split["train"]),
            "val": list(split["val"]),
            "test": list(split["test"]),
            "split_seed": int(seed),
            "split_mode": split_mode,
        }
    raise ValueError(f"Unknown ROBUSTNESS_SPLIT_MODE: {split_mode}")


def build_seed_datasets(seed, split_mode=ROBUSTNESS_SPLIT_MODE, fold=ACTIVE_FOLD):
    """Build train/validation/test datasets for one robustness seed."""
    split = get_robustness_split(seed, split_mode=split_mode, fold=fold)
    x_mean, x_std, a_mean, a_std = compute_norm_stats(split["train"], GRAPHS_DIR)
    train_ds = DAICParticipantDataset(
        split["train"], GRAPHS_DIR, WAV2VEC_DIR, edge_index,
        x_mean, x_std, a_mean, a_std,
        window_frames=WINDOW_FRAMES,
        stride_frames=STRIDE_FRAMES,
        participant_windows=PARTICIPANT_WINDOWS,
        augment=True,
    )
    val_ds = DAICParticipantDataset(
        split["val"], GRAPHS_DIR, WAV2VEC_DIR, edge_index,
        x_mean, x_std, a_mean, a_std,
        window_frames=WINDOW_FRAMES,
        stride_frames=STRIDE_FRAMES,
        participant_windows=PARTICIPANT_WINDOWS,
        augment=False,
    )
    test_ds = DAICParticipantDataset(
        split["test"], GRAPHS_DIR, WAV2VEC_DIR, edge_index,
        x_mean, x_std, a_mean, a_std,
        window_frames=WINDOW_FRAMES,
        stride_frames=STRIDE_FRAMES,
        participant_windows=PARTICIPANT_WINDOWS,
        augment=False,
    )
    return split, train_ds, val_ds, test_ds


def evaluate_model_point_estimate(model, loader, split_name, model_name, device):
    """Evaluate a model without bootstrap so multi-seed runs stay lightweight."""
    preds, labels, pids = collect_scores(model, loader, device)
    report = regression_report(labels, preds)
    return {
        "split": split_name,
        "model": model_name,
        "participant_mae": report["mae"],
        "participant_rmse": report["rmse"],
        "participant_pseudo_macro_f1": report["pseudo_macro_f1"],
        "pred_mean": float(np.mean(preds)),
        "pred_std": float(np.std(preds)),
        "label_mean": float(np.mean(labels)),
        "label_std": float(np.std(labels)),
        "n_participants": int(len(labels)),
    }


def load_best_model_for_robustness(model_name, checkpoint_path, fallback_model):
    """Load the best checkpoint when available; otherwise return the in-memory model."""
    if checkpoint_path and os.path.exists(checkpoint_path):
        if model_name == "baseline":
            return BaselineParticipantPHQ8Regressor.load_from_checkpoint(checkpoint_path, strict=False)
        if model_name == "wav2vec":
            return Wav2VecParticipantPHQ8Regressor.load_from_checkpoint(checkpoint_path, strict=False)
    return fallback_model


def finish_active_wandb_run():
    """Close an active W&B run without failing the notebook when W&B is disabled."""
    try:
        if wandb.run is not None:
            wandb.finish()
    except Exception as exc:
        print(f"[WARN] wandb.finish() failed: {exc}")


def log_robustness_summary_to_wandb(detail_path, summary_path, robustness_df, summary_df, group_name):
    """Log the multi-seed CSV files and summary tables to a final W&B run."""
    if not ROBUSTNESS_USE_WANDB:
        return None
    finish_active_wandb_run()
    run = safe_wandb_init(
        project=ROBUSTNESS_WANDB_PROJECT,
        group=group_name,
        name="version6_robustness_summary",
        job_type="robustness_summary",
        config={
            "version": "Version 6",
            "study": "multi_seed_robustness",
            "seeds": ROBUSTNESS_SEEDS,
            "models": ROBUSTNESS_MODEL_NAMES,
            "split_mode": ROBUSTNESS_SPLIT_MODE,
            "max_epochs": ROBUSTNESS_MAX_EPOCHS,
            "patience": ROBUSTNESS_PATIENCE,
        },
    )
    run.log({
        "robustness/detail_table": wandb.Table(dataframe=robustness_df),
        "robustness/summary_table": wandb.Table(dataframe=summary_df),
    })
    artifact = wandb.Artifact(
        name="version6_robustness_multiseed_results",
        type="robustness_results",
        metadata={
            "version": "Version 6",
            "split_mode": ROBUSTNESS_SPLIT_MODE,
            "n_seeds": len(ROBUSTNESS_SEEDS),
            "models": ROBUSTNESS_MODEL_NAMES,
        },
    )
    artifact.add_file(str(detail_path))
    artifact.add_file(str(summary_path))
    run.log_artifact(artifact)
    wandb.finish()
    return run


## Robustness Training Loop

Run the next cell to train each model once per seed. The results are written to CSV files under `robustness/` and summarized by mean and standard deviation across seeds.


In [ ]:
# ============================================================
# RUN ROBUSTNESS EXPERIMENTS
# ============================================================
def train_one_seed_model(model_name, seed, train_loader, val_loader, split, fold=ACTIVE_FOLD, group_name=ROBUSTNESS_WANDB_GROUP):
    """Train one model for one seed and return the trained artifact metadata."""
    set_experiment_seed(seed)
    model = build_model(model_name)
    run_name = f"version6_robustness_{model_name}_seed{seed}_fold{fold}"
    ckpt_dir = ROBUSTNESS_OUTPUT_DIR / "checkpoints" / f"seed_{seed}" / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    wandb_run = None
    if ROBUSTNESS_USE_WANDB:
        finish_active_wandb_run()
        wandb_run = safe_wandb_init(
            project=ROBUSTNESS_WANDB_PROJECT,
            group=group_name,
            name=run_name,
            job_type="robustness_training",
            tags=["version6", "robustness", "multi_seed", model_name, f"seed_{seed}"],
            config={
                "version": "Version 6",
                "study": "multi_seed_robustness",
                "model": model_name,
                "seed": int(seed),
                "fold": int(fold),
                "split_mode": split["split_mode"],
                "split_seed": int(split["split_seed"]),
                "train_participants": len(split["train"]),
                "val_participants": len(split["val"]),
                "test_participants": len(split["test"]),
                "objective": "PHQ8_score_regression",
                "training_unit": "participant",
                "participant_windows": PARTICIPANT_WINDOWS,
                "window_frames": WINDOW_FRAMES,
                "stride_frames": STRIDE_FRAMES,
                "batch_size": BATCH_SIZE,
                "max_epochs": ROBUSTNESS_MAX_EPOCHS,
                "patience": ROBUSTNESS_PATIENCE,
                "monitor": "val_participant_mae",
            },
        )
        wandb_run.config.update({
            f"{model_name}_hparams": dict(model.hparams),
            f"{model_name}_n_params": sum(p.numel() for p in model.parameters()),
        })

    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        dirpath=str(ckpt_dir),
        monitor="val_participant_mae",
        mode="min",
        save_top_k=1,
        filename=f"{run_name}-{{epoch:02d}}-mae{{val_participant_mae:.3f}}",
        auto_insert_metric_name=False,
    )
    early_stopping = pl.callbacks.EarlyStopping(
        monitor="val_participant_mae",
        patience=ROBUSTNESS_PATIENCE,
        mode="min",
        min_delta=0.01,
        verbose=True,
    )
    csv_logger = CSVLogger(
        save_dir=str(ROBUSTNESS_OUTPUT_DIR / "logs"),
        name=run_name,
    )
    loggers = [csv_logger]
    if wandb_run is not None:
        loggers.append(WandbLogger(experiment=wandb_run, prefix=model_name))

    precision = "16-mixed" if torch.cuda.is_available() else "32"
    trainer = pl.Trainer(
        max_epochs=ROBUSTNESS_MAX_EPOCHS,
        accelerator="auto",
        devices=1,
        log_every_n_steps=5,
        logger=loggers,
        callbacks=[checkpoint_callback, early_stopping],
        gradient_clip_val=1.0,
        precision=precision,
        deterministic=True,
    )
    trainer.fit(model, train_loader, val_loader)
    best_score = checkpoint_callback.best_model_score
    best_ckpt = checkpoint_callback.best_model_path

    if wandb_run is not None and best_ckpt and os.path.exists(best_ckpt):
        ckpt_artifact = wandb.Artifact(
            name=f"version6_robustness_{model_name}_seed{seed}_fold{fold}_ckpt",
            type="model",
            metadata={
                "model": model_name,
                "seed": int(seed),
                "fold": int(fold),
                "best_val_participant_mae": float(best_score) if best_score is not None else None,
            },
        )
        ckpt_artifact.add_file(best_ckpt)
        wandb_run.log_artifact(ckpt_artifact)

    return {
        "model": model,
        "trainer": trainer,
        "best_ckpt": best_ckpt,
        "best_val_mae": float(best_score) if best_score is not None else np.nan,
        "wandb_run": wandb_run,
    }


def run_seed_robustness_study(
    seeds=ROBUSTNESS_SEEDS,
    model_names=ROBUSTNESS_MODEL_NAMES,
    split_mode=ROBUSTNESS_SPLIT_MODE,
    fold=ACTIVE_FOLD,
):
    """Run the full multi-seed robustness study and return detailed and summary tables."""
    rows = []
    started = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    group_name = f"{ROBUSTNESS_WANDB_GROUP}_{started}"
    for seed in seeds:
        print(f"\n================ ROBUSTNESS SEED {seed} ================")
        set_experiment_seed(seed)
        split, seed_train_ds, seed_val_ds, seed_test_ds = build_seed_datasets(seed, split_mode=split_mode, fold=fold)
        seed_train_loader, seed_val_loader, seed_test_loader = make_loaders(
            seed_train_ds,
            seed_val_ds,
            seed_test_ds,
            batch_size=BATCH_SIZE,
        )
        for model_name in model_names:
            print(f"\n[ROBUSTNESS] seed={seed} model={model_name} split_mode={split_mode} fold={fold}")
            artifact = train_one_seed_model(
                model_name,
                seed,
                seed_train_loader,
                seed_val_loader,
                split=split,
                fold=fold,
                group_name=group_name,
            )
            eval_model = load_best_model_for_robustness(model_name, artifact["best_ckpt"], artifact["model"]).to(DEVICE)
            seed_rows = []
            for split_name, loader in [("val", seed_val_loader), ("test_external_dev", seed_test_loader)]:
                metrics = evaluate_model_point_estimate(eval_model, loader, split_name, model_name, DEVICE)
                metrics.update({
                    "seed": int(seed),
                    "fold": int(fold),
                    "split_mode": split_mode,
                    "split_seed": int(split["split_seed"]),
                    "train_participants": len(split["train"]),
                    "val_participants": len(split["val"]),
                    "test_participants": len(split["test"]),
                    "best_val_mae_checkpoint": artifact["best_val_mae"],
                    "checkpoint": artifact["best_ckpt"],
                    "run_started": started,
                    "wandb_group": group_name,
                })
                rows.append(metrics)
                seed_rows.append(metrics)
                print(
                    f"{model_name} | seed={seed} | {split_name}: "
                    f"MAE={metrics['participant_mae']:.4f}, "
                    f"RMSE={metrics['participant_rmse']:.4f}, "
                    f"pseudo-F1={metrics['participant_pseudo_macro_f1']:.4f}"
                )

            if artifact["wandb_run"] is not None:
                payload = {}
                for row in seed_rows:
                    prefix = f"robustness/{row['split']}"
                    payload[f"{prefix}/participant_mae"] = row["participant_mae"]
                    payload[f"{prefix}/participant_rmse"] = row["participant_rmse"]
                    payload[f"{prefix}/participant_pseudo_macro_f1"] = row["participant_pseudo_macro_f1"]
                    payload[f"{prefix}/pred_mean"] = row["pred_mean"]
                    payload[f"{prefix}/pred_std"] = row["pred_std"]
                    payload[f"{prefix}/n_participants"] = row["n_participants"]
                artifact["wandb_run"].log(payload)
                wandb.finish()

            del eval_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    robustness_df = pd.DataFrame(rows)
    detail_path = ROBUSTNESS_OUTPUT_DIR / f"version6_robustness_seed_results_{started}.csv"
    latest_detail_path = ROBUSTNESS_OUTPUT_DIR / "version6_robustness_seed_results_latest.csv"
    robustness_df.to_csv(detail_path, index=False)
    robustness_df.to_csv(latest_detail_path, index=False)

    metric_cols = ["participant_mae", "participant_rmse", "participant_pseudo_macro_f1"]
    summary_df = (
        robustness_df
        .groupby(["model", "split"])[metric_cols]
        .agg(["mean", "std", "min", "max", "count"])
        .reset_index()
    )
    summary_df.columns = ["_".join(col).rstrip("_") if isinstance(col, tuple) else col for col in summary_df.columns]
    summary_path = ROBUSTNESS_OUTPUT_DIR / f"version6_robustness_seed_summary_{started}.csv"
    latest_summary_path = ROBUSTNESS_OUTPUT_DIR / "version6_robustness_seed_summary_latest.csv"
    summary_df.to_csv(summary_path, index=False)
    summary_df.to_csv(latest_summary_path, index=False)

    log_robustness_summary_to_wandb(detail_path, summary_path, robustness_df, summary_df, group_name)

    print(f"\n[SAVED] Detailed robustness results: {detail_path}")
    print(f"[SAVED] Robustness summary: {summary_path}")
    display(robustness_df)
    display(summary_df)
    return robustness_df, summary_df


robustness_df, robustness_summary = run_seed_robustness_study()


## Robustness Visualization

The following cell visualizes the distribution of validation and external-test metrics across seeds.


In [ ]:
# ============================================================
# ROBUSTNESS VISUALIZATION
# ============================================================
if "robustness_df" not in globals():
    latest = ROBUSTNESS_OUTPUT_DIR / "version6_robustness_seed_results_latest.csv"
    robustness_df = pd.read_csv(latest)

plot_df = robustness_df.copy()
metric_specs = [
    ("participant_mae", "MAE"),
    ("participant_rmse", "RMSE"),
    ("participant_pseudo_macro_f1", "Pseudo macro F1"),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (metric, title) in zip(axes, metric_specs):
    sns.boxplot(data=plot_df, x="model", y=metric, hue="split", ax=ax)
    sns.stripplot(data=plot_df, x="model", y=metric, hue="split", dodge=True, alpha=0.65, color="black", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Model")
    ax.set_ylabel(title)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], title="Split")
    ax.grid(True, alpha=0.25)
plt.tight_layout()
plot_path = ROBUSTNESS_OUTPUT_DIR / "version6_robustness_seed_metric_distribution.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[SAVED] {plot_path}")


## Reporting Template

Use the generated summary table to report whether the final models are stable across seeds. In the thesis, prioritize the mean and standard deviation of MAE, RMSE, and pseudo macro F1 on the validation and external development-test splits.


In [ ]:
# ============================================================
# LATEX-FRIENDLY SUMMARY TABLE
# ============================================================
if "robustness_summary" not in globals():
    latest = ROBUSTNESS_OUTPUT_DIR / "version6_robustness_seed_summary_latest.csv"
    robustness_summary = pd.read_csv(latest)

latex_cols = [
    "model",
    "split",
    "participant_mae_mean",
    "participant_mae_std",
    "participant_rmse_mean",
    "participant_rmse_std",
    "participant_pseudo_macro_f1_mean",
    "participant_pseudo_macro_f1_std",
    "participant_mae_count",
]
latex_summary = robustness_summary[latex_cols].copy()
for col in latex_summary.columns:
    if col not in ["model", "split", "participant_mae_count"]:
        latex_summary[col] = latex_summary[col].map(lambda x: f"{x:.4f}")
display(latex_summary)
print(latex_summary.to_latex(index=False, escape=False))
